In [ ]:
# Load a COCO-pretrained YOLOv8n model
#that will create a file called yolov8n that we will use later for prediction
model = YOLO("yolov8n.pt", "v8")

# Display model information (optional)
model.info()

In [ ]:
import numpy as np
from ultralytics import YOLO
import random
import cv2
import threading
import server


In [2]:
# Initialize voice engine
engine = pyttsx3.init()
voices = engine.getProperty('voices')
for i, voice in enumerate(voices):
    print(f"{i}: {voice.name} ({voice.languages})")
engine.setProperty('voice', voices[1].id)
engine.setProperty('rate', 200)  # Speed of speech
engine.setProperty('volume', 1)  # Volume level (0.0 to 1.0)

0: Microsoft Hortense Desktop - French ([])
1: Microsoft Zira Desktop - English (United States) ([])


In [3]:
my_file = open("objects.txt", "r")# this file have all objects 

# reading the file
data = my_file.read()
# split when newline ('\n') is seen
class_list = data.split("\n")
my_file.close()

In [5]:
print(class_list)


['Chair', 'Lamp', 'TrashBin', 'Tree', 'Car', 'Crosswalk', 'Person', 'Motorcycle', 'Bicycle', 'TraficLight', 'CrossSign', 'Dog', 'Wall', 'Bus', 'Cat', 'Barrier', 'DeliveryBox', 'FireHydrant', 'FallenSign', 'Fence', 'Hole_in_the_road', 'Road_cone', 'Open_manhole', 'Road_workahead', 'shopping_cart']


In [6]:

# Danger élevé
DANGER_ELEVE = {
    "Open_manhole", "Hole_in_the_road", "Car", "Bus", "Motorcycle",
    "Bicycle", "FireHydrant", "Barrier", "Wall", "Fence", "Road_cone",
    "Road_workahead", "TrashBin", "FallenSign"
}

# Danger régulier
DANGER_REGULIER = {
    "Person", "Dog", "Cat", "Crosswalk", "Chair", "Lamp", "TraficLight",
    "CrossSign", "DeliveryBox", "shopping_cart", "Tree"
}

In [7]:
# just random colors for bounding boxes
Boxes = []
for i in range(len(class_list)):
    r = random.randint(0, 255)
    g = random.randint(0, 255)
    b = random.randint(0, 255)
    Boxes.append((b, g, r))

In [ ]:
#resize video frames to optimise the run
frame_wid = 1280
frame_hyt = 720

In [9]:
# Define your custom message base, without distance part:
def get_object_message(class_name):
    messages = {
        "Open_manhole": "Warning! An open manhole is",
        "Hole_in_the_road": "Danger! A hole in the road is",
        "Car": "Warning! A car is",
        "Bus": "Warning! A bus is",
        "Motorcycle": "Warning! A motorcycle is",
        "Bicycle": "Warning! A bicycle is",
        "FireHydrant": "There is a fire hydrant",
        "Barrier": "Warning! A barrier is",
        "Wall": "A wall is",
        "Fence": "A fence is",
        "Road_cone": "Warning! A road cone is",
        "Road_workahead": "Danger! Road work ahead",
        "TrashBin": "A trash bin is",
        "FallenSign": "A fallen sign is",
        "Person": "A person is",
        "Dog": "A dog is",
        "Cat": "A cat is",
        "Crosswalk": "If you want to cross the road. A crosswalk is ",
        "Chair": "A chair is",
        "Lamp": "A lamp post is",
        "TraficLight": "A traffic light is",
        "CrossSign": "A traffic sign is",
        "DeliveryBox": "A delivery box is",
        "shopping_cart": "A shopping cart is",
        "Tree": "A tree is",
    }
    return messages.get(class_name, f"{class_name} detected and is")

In [10]:
import threading
import winsound

def announce_distance(class_name, distance):
    def speak():
        # If object is high danger, play beep first
        if class_name in DANGER_ELEVE:
            winsound.Beep(1000, 200)
            winsound.Beep(1000, 200) 

        base_msg = get_object_message(class_name)
        if distance < 1:
            full_msg = f"{base_msg} in front of you, less than 1 meter away."
        elif 1 <= distance < 2:
            full_msg = f"{base_msg} in front of you, less than 2 meters away."
        elif 2 <= distance < 3:
            full_msg = f"{base_msg} in front of you, about 2 to 3 meters away."
        elif 3 <= distance < 4:
            full_msg = f"{base_msg} in front of you, about 3 meters away."
        elif 4 <= distance < 5:
            full_msg = f"{base_msg} in front of you, about 4 meters away."
        elif 5 <= distance < 6:
            full_msg = f"{base_msg} in front of you, about 5 meters away."
        else:
            full_msg = f"{base_msg} far away, more than 5 meters."

        engine.say(full_msg)
        engine.runAndWait()

    threading.Thread(target=speak).start()


In [11]:
# Ouvrir la caméra
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Cannot open camera")
    exit()

In [14]:
import cv2
import time
from ultralytics import YOLO

# Dictionnaire des largeurs réelles (en mètres)
REAL_WIDTHS = {
    "Chair": 0.5, "Lamp": 0.3, "TrashBin": 0.4, "Tree": 0.5, "Car": 1.8,
    "Crosswalk": 2.5, "Person": 0.5, "Motorcycle": 0.8, "Bicycle": 0.6,
    "TraficLight": 0.3, "CrossSign": 0.5, "Dog": 0.4, "Wall": 2.0,
    "Bus": 2.5, "Cat": 0.25, "Barrier": 1.2, "DeliveryBox": 0.5,
    "FireHydrant": 0.4, "FallenSign": 0.6, "Fence": 2.0,
    "Hole_in_the_road": 0.7, "Road_cone": 0.3, "Open_manhole": 0.6,
    "Road_workahead": 1.0, "shopping_cart": 0.55
}


FOCAL_LENGTH = 462
announcement_delay = 5  # secondes
last_announced_class = None
last_announcement_time = 0


# Charger le modèle
model = YOLO("./Model.pt", "v8")
cap = cv2.VideoCapture(0)

# Liste des couleurs des bounding boxes
Boxes = [(255, 0, 0)] * 100  # Modifier selon besoins
class_list = list(REAL_WIDTHS.keys())

while True:
    ret, frame = cap.read()
    if not ret:
        print("Can't receive frame. Exiting ...")
        break

    overlay = frame.copy()
    cv2.putText(overlay, "Montrez un objet pour détecter...", (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

    detect_params = model.predict(source=[frame], conf=0.45, save=False)
    DP = detect_params[0].cpu().numpy()

    if len(DP) != 0 and len(detect_params[0].boxes) != 0:
        detected_objects = []
        boxes = detect_params[0].boxes

        for i in range(len(boxes)):
            box = boxes[i]
            clsID = int(box.cls.cpu().numpy()[0])
            conf = round(float(box.conf.cpu().numpy()[0]), 3)
            bb = box.xyxy.cpu().numpy()[0]

            class_name = class_list[clsID]
            x1, y1, x2, y2 = map(int, bb)
            object_roi = frame[y1:y2, x1:x2]

            gray = cv2.cvtColor(object_roi, cv2.COLOR_BGR2GRAY)
            _, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY)
            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            if contours:
                largest_contour = max(contours, key=cv2.contourArea)
                rect = cv2.minAreaRect(largest_contour)
                object_width_pixels = min(rect[1])

                object_width_real = REAL_WIDTHS.get(class_name, None)
                if object_width_real and object_width_pixels > 0:
                    distance = (object_width_real * FOCAL_LENGTH) / object_width_pixels
                    detected_objects.append((class_name, distance, bb, clsID, conf))

        # Priorité danger élevé
        danger_eleve = [obj for obj in detected_objects if obj[0] in DANGER_ELEVE]
        if danger_eleve:
            closest = min(danger_eleve, key=lambda x: x[1])
        else:
            closest = min(detected_objects, key=lambda x: x[1])

        class_name, distance, bb, clsID, conf = closest
        x1, y1, x2, y2 = map(int, bb)

        # Annonce : pas de délai si c'est une classe différente
        current_time = time.time()
        if class_name != last_announced_class or (current_time - last_announcement_time > announcement_delay):
            announce_distance(class_name, distance)
            last_announced_class = class_name
            last_announcement_time = current_time

        # Affichage
        cv2.rectangle(frame, (x1, y1), (x2, y2), Boxes[clsID], 3)
        cv2.putText(frame, f"{class_name} {conf}%", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
        cv2.putText(frame, f"Distance: {round(distance, 2)} m", (x1, y1 + 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    # Affichage final
    cv2.imshow("ObjectDetection", frame if len(DP) != 0 else overlay)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break
    if cv2.getWindowProperty("ObjectDetection", cv2.WND_PROP_VISIBLE) < 1:
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)



0: 480x640 (no detections), 105.3ms
Speed: 6.8ms preprocess, 105.3ms inference, 224.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 101.5ms
Speed: 5.4ms preprocess, 101.5ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 101.8ms
Speed: 3.7ms preprocess, 101.8ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 101.9ms
Speed: 3.3ms preprocess, 101.9ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 100.8ms
Speed: 14.0ms preprocess, 100.8ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 101.8ms
Speed: 3.8ms preprocess, 101.8ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 102.0ms
Speed: 2.3ms preprocess, 102.0ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 102.0ms
Speed: 3.2ms pr

Exception in thread Thread-10 (speak):
Traceback (most recent call last):
  File "C:\Users\Spectre\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "c:\Users\Spectre\Desktop\ML\PI\YoloEnv\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\Spectre\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Spectre\AppData\Local\Temp\ipykernel_3268\2751592207.py", line 28, in speak
  File "c:\Users\Spectre\Desktop\ML\PI\YoloEnv\Lib\site-packages\pyttsx3\engine.py", line 180, in runAndWait
    raise RuntimeError('run loop already started')
RuntimeError: run loop already started



0: 480x640 1 Person, 101.6ms
Speed: 3.4ms preprocess, 101.6ms inference, 4.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 101.8ms
Speed: 3.1ms preprocess, 101.8ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 101.4ms
Speed: 6.6ms preprocess, 101.4ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 102.4ms
Speed: 3.5ms preprocess, 102.4ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 101.9ms
Speed: 2.1ms preprocess, 101.9ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 102.2ms
Speed: 2.4ms preprocess, 102.2ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 102.1ms
Speed: 3.8ms preprocess, 102.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 101.8ms
Speed: 2.7ms preprocess, 

-1